In [ ]:
# ---------------- code impératif avec sqlite3 -------------
"""
SQLITE3: BDD relationnelle avec un dialecte SQL simple
BDD en un seul fichier ou chargé en RAM
pas de host / port / user / password / droits
le module sqlite3 python est dans la lib standard

sqlite3 utilise la DbAPI python pour tous les clients de BDD en python
- méthode connect() => connexion (création ou connexion sur la bdd)
- méthode cursor()  => prompt sur la connexion
- méthode execute() => exécuter une requête SQL sur le cursor
- méthode fetch()   => faire sortir les enregistrements attendus
- méthode close()   => fermer la cnx
---------------- gérer les transactions -----------------
- méthode begin()   => commence une transaction
- methode commit()  => confitmer une tr
- méthode rollback()=> annuler une tr
"""

import sqlite3

conn = sqlite3.connect("dns.db")
cur  = conn.cursor()
req = "SELECT SQLITE_VERSION()"
cur.execute(req)
row  = cur.fetchone()
print(row)
conn.close()

In [ ]:
# ----------------- une connexion s'ouvre et se ferme ------------------
# exécuter un script SQL pour créer les tables de la bdd
with sqlite3.connect("dns.db") as conn:
    cur  = conn.cursor()
    with open("./domain_names_sqlite3.sql") as f:
        script = f.read()
    cur.executescript(script)

    # test de l'insertion
    req = "SELECT * FROM pays"
    cur.execute(req)
    rows = cur.fetchall()
    print(f"lignes changées: {len(rows)}")

    # alternative: demander les lignes modifiées
    # print(f"lignes changées: {cur.rowcount}")


In [ ]:
# ----------------------- insérer 100k lignes à partir d'un csv -----------------
"""
1. collecter le nom de domaine et le pays en 2 caractère depuis le fichier dns_100000.csv
2. insérer les données dans la table domain_name
   requête INSERT INTO domain_name (name, iso2) VALUES (?, ?)
   en utilisant la méthode executemany()
   HINT: commencer avec 100 lignes et si çà fonctionne faites des batches de 10000
3. vérifier que les données ont été insérées.
"""
import csv

data = []
with open("./data/dns_100000.csv", mode="r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter=";")
    next(reader)
    for i, row in enumerate(reader, start=1):
        # if i > 100: break
        data.append(row[:2])

## fabriquer une grande requête INSERT à partir d'une collection de données
## (batching)
# for i in range(len(data)):
#     data[i] = "','".join(data[i])
# data = "'),('".join(data)
# version map: data = "'),('".join(list(map(lambda r: "','".join(r), data)))




with sqlite3.connect("dns.db") as conn:
    cur  = conn.cursor()
    # version avec 100k x 1 requête
    # req = "INSERT INTO domain_name (name, iso2) VALUES (?, ?)"
    # cur.executemany(req, data)

    # version 100k valeurs dans 1 requête

    # version 10k valeurs x 10 requêtes: batching
    while data:
        sample = data[:10**4]
        sample = "'),('".join(list(map(lambda r: "','".join(r), sample)))
        req = f"INSERT INTO domain_name (name, iso2) VALUES ('{ sample }')"
        cur.execute(req)
        data  = data[10**4:]
        print(f"lignes changées: {cur.rowcount}")



In [ ]:
# ----------------- client sqlite3 en poo et gestionnaire de contexte
from pathlib import Path
# créer des annotations plus précises
from typing import List

class SqliteClient:
    def __init__(self, db_path: str | Path):
        self.db_path = db_path

    def execute_script(self, script_path: str | Path):
        with sqlite3.connect(self.db_path) as conn:
            cur  = conn.cursor()
            with open(script_path) as f:
                script = f.read()
            cur.executescript(script)

    def count(self, table: str) -> int:
        with sqlite3.connect(self.db_path) as conn:
            cur  = conn.cursor()
            req = f"SELECT COUNT(1) FROM {table}"
            cur.execute(req)
            return cur.fetchone()[0]

    def insert(self, table: str, data: List[list], fields: list = None) -> int:
        with sqlite3.connect(self.db_path) as conn:
            cur  = conn.cursor()
            fields_str = "(" + ",".join(fields) + ")" if fields is not None else ""
            req = f"INSERT INTO {table} {fields_str} VALUES (?, ?)"
            cur.executemany(req, data)
            return cur.rowcount


db = SqliteClient("dns.db")
db.execute_script("./domain_names_sqlite3.sql")
print(db.count("pays"))
data = [("MZ", "Mozambique")]
# nb = db.insert("pays", fields=["iso2", "name"], data=data)
nb = db.insert("pays", data)
print(nb)


In [ ]:
# ------------------ client avec gestionnaire de contexte ----------
from pathlib import Path
# créer des annotations plus précises
from typing import List

class SqliteClient:
    def __init__(self, db_path: str | Path):
        self.db_path = db_path
        # self.conn = sqlite3.connect(self.db_path)

    def __enter__(self):
        self.conn = sqlite3.connect(self.db_path)
        # "as xxx"
        return self

    def __exit__(self, exc_type, exc, tb):
        self.conn.close()

    def execute_script(self, script_path: str | Path):
        cur  = self.conn.cursor()
        with open(script_path) as f:
            script = f.read()
        cur.executescript(script)

    def count(self, table: str) -> int:
        cur  = self.conn.cursor()
        req = f"SELECT COUNT(1) FROM {table}"
        cur.execute(req)
        return cur.fetchone()[0]

    def insert(self, table: str, data: List[list], fields: list = None) -> int:
        cur  = self.conn.cursor()
        fields_str = "(" + ",".join(fields) + ")" if fields is not None else ""
        req = f"INSERT INTO {table} {fields_str} VALUES (?, ?)"
        cur.executemany(req, data)
        return cur.rowcount

    # def close(self):
    #     self.conn.close()


# db = SqliteClient("dns.db")
with SqliteClient("dns.db") as db:
    db.execute_script("./domain_names_sqlite3.sql")
    print(db.count("pays"))
    data = [("MZ", "Mozambique")]
    # nb = db.insert("pays", fields=["iso2", "name"], data=data)
    nb = db.insert("pays", data)
    print(nb)
    # db.close()



In [ ]:
# -------------------- initiation à sqlAlchemy (véritable ORM) ---------------

# pip install SQLAlchemy

# connexion et session pour pooling
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, Session

# définition des modèles
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
# champs SQL
from sqlalchemy import String, Enum, Integer, Float, DateTime, Text, ForeignKey, func

from pathlib import Path

# -------------------------------- GLOBALS ------------------------------------

DB_PATH = "./dns.db"
# chaîne de cnx :URI
DB_URI = f"sqlite:///{DB_PATH}"


engine = create_engine(
  DB_URI,
  # par défaut: on peut réutiliser le même thread pour une nouvelle cnx => pooling possible
  connect_args={"check_same_thread": False},
  # voir les requêtes SQL pour debug
  #echo=True
)

# factory de sessions
SessionLocal = sessionmaker(autoflush=False, autocommit=False, bind=engine)

# --------------------------- MODELES --------------------------------------------

class Base(DeclarativeBase):
  """classe de base pour tous les modèles (tables)"""
  pass

class DomainName(Base):
  __tablename__ = "domain_name"

  id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
  iso2: Mapped[str] = mapped_column(String(2), nullable=False)
  name: Mapped[str] = mapped_column(String(100), unique=True, nullable=False, index=True)

class Pays(Base):
  __tablename__ = "pays"
  iso2: Mapped[str] = mapped_column(String(2), primary_key=True)
  name: Mapped[str] = mapped_column(String(100), unique=True, nullable=False)

# ------------------------------- init_db --------------------------------------------------


def init_db():
  import re
  import csv

  Base.metadata.create_all(bind=engine)

  with SessionLocal() as session:
    # Import pays depuis le script SQL
    sql_path = "./domain_names_sqlite3.sql"
    with open(sql_path, "r", encoding="utf-8") as f:
      sql_content = f.read()

    match = re.search(r"INSERT INTO `pays` VALUES (.+);", sql_content)
    if match:
      pays_values = re.findall(r"\('([^']+)','([^']+)'\)", match.group(1))
      for iso2, name in pays_values:
        # Pays(iso2=iso2, name=name)  => objet enregistrement de table
        # cet objet a une réalité en python et aussi dans la base
        # il faut synchroniser régulièrement les objet py et sql flush() commit()
        session.merge(Pays(iso2=iso2, name=name))

    # Import domain_name depuis dns_10.csv
    csv_path = "./data/dns_100000.csv"
    with open(csv_path, mode="r", encoding="utf-8") as f:
      reader = csv.DictReader(f, delimiter=";")
      for row in reader:
        session.add(DomainName(name=row["Nom de domaine"], iso2=row["Pays BE"]))

    session.commit()

init_db()

In [ ]:
# ------------------------- usage ----------------------------------------
from sqlalchemy import select, func

with SessionLocal() as session:
  # SELECT * FROM domain_name
  dns_count = session.execute(select(func.count()).select_from(DomainName)).scalar()
  print(f"Nombre de domaines: {dns_count}")